# Lesson 03 - Agentic Design Patterns

Sa leksyong ito, tatalakayin natin ang tatlong pundamental na design patterns para sa paggawa ng epektibong AI agents:

1. **Clear Agent Instructions** — Paglikha ng tumpak at malinaw na mga prompt na nagtatakda ng papel ng agent upang gabayan ang kilos nito
2. **Structured Output with Pydantic Models** — Pagtitiyak na ang mga agent ay nagbabalik ng prediktableng at na-validate na data
3. **Single Responsibility Agents** — Pagdidisenyo ng mga nakatuon na agent na bawat isa ay mahusay sa isang gawain lamang

Ilalapat natin ang bawat pattern sa isang **travel destination recommender** na senaryo, kung saan unti-unting bubuo tayo ng sistema na makakapagsuggest ng mga destinasyon, magche-check ng availability, at hahawakan ang mga logistika.


## Pag-setup


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

## Pattern 1: Malinaw na Mga Panuto para sa Ahente

Ang pinakaepektibong pattern ay ang pinaka-simple rin: pagsulat ng malinaw, detalyadong mga panuto para sa iyong ahente.

Ang magagandang panuto ay naglalarawan ng:
- **Sino** ang ahente (persona at tono)
- **Ano** ang dapat gawin nito (mga responsibilidad hakbang-hakbang)
- **Paano** ito dapat kumilos (mga limitasyon at estilo)

Sa ibaba, lumikha tayo ng travel concierge agent na may malinaw na mga panuto na humuhubog sa bawat tugon na ginagawa nito.


In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

agent = await provider.create_agent(
    name="TravelConcierge",
    instructions="""You are a luxury travel concierge named Alex. Your role is to:
1. Understand the traveler's preferences (budget, climate, activities)
2. Check destination availability before making recommendations
3. Provide detailed, personalized travel suggestions
4. Always mention visa requirements and best travel seasons
Be warm, professional, and enthusiastic about travel.""",
)

response = await agent.run(
    "I'd love a week-long vacation somewhere with great food and history. Budget around $2500."
)
print(response)

## Pattern 2: Nakabalangkas na Output gamit ang mga Pydantic Model

Ang malayang teksto ay kapaki-pakinabang para sa pag-uusap, ngunit kailangan ng mga downstream na sistema ng nakaayos na datos.  
Sa pamamagitan ng pagpapares ng **mga Pydantic model** sa isang **function ng tool**, maaari nating:

- Tukuyin ang eksaktong schema para sa output ng ahente  
- Awtomatikong beripikahin ang mga tugon  
- Isama ang mga resulta ng ahente sa lohika ng aplikasyon nang maaasahan  

Ipinapakilala rin namin ang isang tool na nagbabalik ng mga detalye ng patutunguhan upang maging batayan ng ahente ang mga rekomendasyon nito sa totoong datos.


In [ ]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str


@tool(approval_mode="never_require")
def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
    }
    return details.get(destination, f"{destination}: No information available.")


structured_agent = await provider.create_agent(
    name="StructuredTravelExpert",
    instructions="You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool.",
    tools=[get_destination_details],
)

response = await structured_agent.run(
    "Recommend 3 destinations for a culture-loving traveler with a $2500 budget"
)

if response:
    print(response)

## Pattern 3: Mga Ahente na May Tanging Responsibilidad

Ang mga komplikadong gawain ay nakikinabang sa paghahati ng trabaho sa maraming nakatuon na ahente, bawat isa ay may tanging responsibilidad:

- Isang **Eksperto sa Destinasyon** na nakakaalam tungkol sa mga lugar at pagkakaroon
- Isang **Tagaplano ng Logistiko** na humahawak ng mga flight, hotel, at mga itinerary

Ito ay sumasalamin sa prinsipyo ng software engineering na *paghihiwalay ng mga alalahanin* — mas madali ang pagsubok, pagpapanatili, at pagpapabuti ng bawat ahente nang hiwalay.


In [ ]:
destination_agent = await provider.create_agent(
    name="DestinationExpert",
    tools=[get_destination_details],
    instructions="""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that.""",
)

logistics_agent = await provider.create_agent(
    name="LogisticsPlanner",
    instructions="""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that.""",
)

# Step 1: Destination Expert picks the best options
dest_response = await destination_agent.run(
    "I want a week of culture and food for under $2500. Where should I go?"
)
print("=== Destination Expert ===")
print(dest_response)

# Step 2: Logistics Planner builds the trip plan
logistics_response = await logistics_agent.run(
    f"Plan a week-long trip based on this recommendation:\n{dest_response}"
)
print("\n=== Logistics Planner ===")
print(logistics_response)

## Buod

Sa araling ito inapply namin ang tatlong agentic design patterns sa isang travel recommender na senaryo:

| Pattern | Pangunahing Ideya | Benepisyo |
|---|---|---|
| **Malinaw na mga Tagubilin** | Tukuyin ang persona, mga responsibilidad, at mga limitasyon nang maaga | Konsistent, on-brand na kilos ng ahente |
| **Istrakturadong Output** | Gamitin ang mga Pydantic models bilang format ng sagot | Napatunayan, makinang mabasang resulta |
| **Nag-iisang Responsibilidad** | Bigyan ang bawat ahente ng isang nakatuong trabaho | Mas madaling subukan, panatilihin, at pagsamahin |

Ang mga pattern na ito ay natural na nagsasama — maaari mong paghaluin ang malinaw na mga tagubilin sa istrakturadong output sa loob ng isang nag-iisang responsibilidad na ahente upang bumuo ng matibay, handa para sa produksyon na mga sistema.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Paalala**:
Ang dokumentong ito ay isinalin gamit ang AI translation service na [Co-op Translator](https://github.com/Azure/co-op-translator). Bagamat aming pinagsisikapang maging tumpak ang salin, pakatandaan na ang mga awtomatikong pagsasalin ay maaaring magkaroon ng mga pagkakamali o di-tumpak na impormasyon. Ang orihinal na dokumento sa kanyang sariling wika ang dapat ituring na pangunahing sanggunian. Para sa mahahalagang impormasyon, inirerekomenda ang propesyonal na pagsasalin ng tao. Hindi kami mananagot sa anumang hindi pagkakaunawaan o maling interpretasyon na maaaring magmula sa paggamit ng pagsasaling ito.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
